In [ ]:
# Gemma-SEA-LION-v3-9B QLoRA fine-tune, tuned for Kaggle "GPU T4 x2" 
#
# Before running on Kaggle: Settings -> Accelerator -> "GPU T4 x2".

# 1. Install required libraries (latest versions are fine: Gemma 2 is natively supported,
#    so no version pinning or trust_remote_code patches are needed).
#    If you earlier installed the old pinned versions (transformers==4.41.2, trl==0.8.6),
#    this upgrades them. Restart the kernel after installing.
# !pip install -q -U transformers datasets peft bitsandbytes accelerate trl

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTConfig, SFTTrainer

print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {props.name} {props.total_memory / 1e9:.1f} GB")

# Gemma-based models can be gated on Hugging Face. If loading fails with 401/403,
# accept the license on the model page and log in with a token:
# from huggingface_hub import login
# login("hf_xxx...")   # or add HF_TOKEN as a Kaggle secret

# 2. Load and downsample the dataset
dataset = load_dataset("aisyahhrazak/combined-malay-sentiment", split="train")
dataset = dataset.shuffle(seed=42).select(range(10000))
print(dataset.column_names)  

model_id = "aisingapore/Gemma-SEA-LION-v3-9B"  # base model

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. Format the data into an instruction prompt (single "text" column).
#    EOS is appended so the model learns to stop after the label.
def to_prompt(example):
    instruction = "Analyze the sentiment of the following Malay/Manglish text. Classify it as POSITIVE, NEGATIVE, or NEUTRAL."
    sentiment = str(example["sentiment"]).upper()
    text = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{example['text']}\n\n"
        f"### Response:\n{sentiment}{tokenizer.eos_token}"
    )
    return {"text": text}

dataset = dataset.map(to_prompt, remove_columns=dataset.column_names)

# 4. Load model in 4-bit (QLoRA), split across both T4s
# T4 does not support bf16 properly -> float16 + fp16=True
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

# device_map="auto" spreads the layers across both GPUs (model parallel).
# The two GPUs share the memory load, they do not run twice as fast.
# max_memory leaves headroom on each card for activations during training.
n_gpus = torch.cuda.device_count()
max_memory = {i: "13GiB" for i in range(n_gpus)}

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    attn_implementation="eager",
    dtype=torch.float16,   # <-- add this (the T4 has no bf16; Gemma loads as bf16 by default)
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# fp16 training needs trainable weights in float32 because bf16/fp16 grads can't be unscaled
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

model.print_trainable_parameters()
print("Trainable dtypes:", {p.dtype for p in model.parameters() if p.requires_grad})

# 6. Training parameters
training_args = SFTConfig(
    output_dir="./gemma-sealion-sentiment-lora",
    per_device_train_batch_size=4,      # drop to 2 if hit CUDA out-of-memory
    gradient_accumulation_steps=4,      # effective batch size = 16
    learning_rate=1e-4,
    logging_steps=10,
    max_steps=200,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    max_length=384,
    dataset_text_field="text",
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=100,
    report_to="none",
)

# 7. Train
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

# Check what the trainer did to the trainable weights
bad = [n for n, p in trainer.model.named_parameters()
       if p.requires_grad and p.dtype != torch.float32]
print("Non-fp32 trainable params after trainer init:", len(bad), bad[:3])

# Force them back to float32
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

print("Trainable dtypes now:", {p.dtype for p in trainer.model.parameters() if p.requires_grad})

print("Starting LoRA Fine-Tuning...")
trainer.train()

# 8. Save the LoRA adapters
trainer.model.save_pretrained("my-custom-malay-sentiment-lora-gemma9b")
tokenizer.save_pretrained("my-custom-malay-sentiment-lora-gemma9b")
print("LoRA weights saved successfully")

In [ ]:
# save model as zip
import shutil
shutil.make_archive("/kaggle/working/lora_gemma9b", "zip",
                    "/kaggle/working/my-custom-malay-sentiment-lora-gemma9b")

In [ ]:
# to evaluate the fine tuned model

import torch
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report

N = 200  # number of test rows

# Same shuffle as training (seed=42), but rows 10000+ were never trained on
raw = load_dataset("aisyahhrazak/combined-malay-sentiment", split="train").shuffle(seed=42)
test = raw.select(range(10000, 10000 + N))

labels = sorted({str(x).upper() for x in raw.select(range(10000))["sentiment"]})
true = [str(x).upper() for x in test["sentiment"]]
texts = test["text"]

print("Labels:", labels)
print("Test label counts:", Counter(true))
print("Majority-class accuracy:", max(Counter(true).values()) / N)

def make_prompt(text):  # must match the training format exactly
    instruction = "Analyze the sentiment of the following Malay/Manglish text. Classify it as POSITIVE, NEGATIVE, or NEUTRAL."
    return f"### Instruction:\n{instruction}\n\n### Input:\n{text[:600]}\n\n### Response:\n"

@torch.no_grad()
def predict(model, texts, bs=8):
    model.eval()
    tokenizer.padding_side = "left"
    preds = []
    for i in range(0, len(texts), bs):
        batch = [make_prompt(t) for t in texts[i:i + bs]]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to("cuda:0")
        out = model.generate(**enc, max_new_tokens=6, do_sample=False,
                             use_cache=True, pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for g in gen:
            g = g.strip().upper()
            preds.append(next((l for l in labels if g.startswith(l)), "UNPARSEABLE"))
    return preds


In [ ]:
# to compare original fined tuned with my custom fine tuned model

model = trainer.model  # fine-tuned model, still in memory

ft_preds = predict(model, texts)

with model.disable_adapter():   # temporarily turns the LoRA off = original base model
    base_preds = predict(model, texts)

print("Fine-tuned accuracy:", accuracy_score(true, ft_preds))
print("Untrained accuracy: ", accuracy_score(true, base_preds))
print("Untrained unparseable answers:", base_preds.count("UNPARSEABLE"))
print("\nFine-tuned details:\n", classification_report(true, ft_preds, zero_division=0))